In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import optuna
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error

print("✓ Importy załadowane")

# Wczytaj dane z cechami
df = pd.read_csv("C:\/Users\/aiszk\PycharmProjects\PythonProject\data\processed\/features_data.csv", parse_dates=['timestamp'])


✓ Importy załadowane


In [5]:
# Usuń kolumny nieistotne i kategoryczne
drop_cols = ['timestamp', 'meter_id', 'campus_id', 'consumption', 'date_dt', 'id', 'id_building', 'id_campus']
drop_cols = [col for col in drop_cols if col in df.columns]

X = df.drop(drop_cols, axis=1)
y = df['consumption']

# Usuń kolumny kategoryczne (stringi)
categorical_cols = X.select_dtypes(include=['object']).columns
X = X.drop(categorical_cols, axis=1)

# Uzupełnij braki danymi średnimi
X = X.fillna(X.mean())

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Split na zbiór treningowy i testowy – użyj temporalnie jeśli chcesz
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


X shape: (4484722, 48)
y shape: (4484722,)


In [6]:
def objective(trial):
    alpha = trial.suggest_loguniform('alpha', 1e-4, 10.0)
    max_iter = trial.suggest_int('max_iter', 1000, 10000)

    model = Ridge(alpha=alpha, max_iter=max_iter)

    # Walidacja krzyżowa 3-krotna
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='neg_root_mean_squared_error').mean()
    return -score


In [7]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print(f"Najlepsze parametry: {study.best_params}")
print(f"Najlepsza wartość RMSE CV: {study.best_value:.4f}")


[I 2025-11-19 23:34:08,587] A new study created in memory with name: no-name-8af39d52-47a8-4cce-9721-aa945c7cc2e0
<ipython-input-6-680217077d56>:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  alpha = trial.suggest_loguniform('alpha', 1e-4, 10.0)
C:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_ridge.py:147: LinAlgWarning: Ill-conditioned matrix (rcond=3.89787e-20): result may not be accurate.
  return linalg.solve(A, Xy, sym_pos=True,
C:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_ridge.py:147: LinAlgWarning: Ill-conditioned matrix (rcond=1.87531e-21): result may not be accurate.
  return linalg.solve(A, Xy, sym_pos=True,
C:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_ridge.py:147: LinAlgWarning: Ill-conditioned matrix (rcond=1.93236e-21): result may not be accurate.
  retur

Najlepsze parametry: {'alpha': 9.965757309260876, 'max_iter': 8119}
Najlepsza wartość RMSE CV: 1.0171


In [8]:
best_params = study.best_params
best_model = Ridge(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE najlepszego modelu: {test_rmse:.4f}")


C:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_ridge.py:147: LinAlgWarning: Ill-conditioned matrix (rcond=9.13988e-20): result may not be accurate.
  return linalg.solve(A, Xy, sym_pos=True,


Test RMSE najlepszego modelu: 1.0613


In [10]:
# Utwórz folder, jeśli nie istnieje
Path("models").mkdir(parents=True, exist_ok=True)

import joblib
joblib.dump(best_model, "models/ridge_optimized.joblib")
print("✓ Model zapisany")


✓ Model zapisany
